# Entrenamiento de locomoción Unitree G1 — LocoMuJoCo (PFI) — Versión WSL + GPU

Basado en la versión LOCAL (CPU) que dio buen resultado -- misma lógica de entrenamiento por **etapas idempotentes** con guardado y retomado automático -- ajustada para GPU real (GTX 1650, 4GB VRAM) en WSL2/Ubuntu.

## Prerrequisitos (ya resueltos en esta máquina, dejar como referencia)
- WSL2 con Ubuntu, GPU visible vía `nvidia-smi` (confirmado).
- Entorno virtual `~/venv_g1_locomotion` con Python 3.12, `jax[cuda12]`, `flax` actualizado, `loco-mujoco` desde GitHub, `wandb`, `jupyterlab` instalados.
- Diagnóstico por capas (`verify_env.py`) ya confirmado OK, incluyendo `gpu_compute` con `XLA_PYTHON_CLIENT_PREALLOCATE=false`.

## Ajuste clave respecto a la versión CPU/Colab: memoria de GPU limitada (4GB)
Tu placa tiene mucha menos VRAM que la T4 de Colab (15GB). Este notebook parte con `num_envs` bajo por defecto y `XLA_PYTHON_CLIENT_PREALLOCATE=false` seteado siempre -- ajustá `NUM_ENVS` hacia arriba de a poco si ves que sobra memoria, no al revés.

## 0. Configurar la ruta de tu repositorio

Ajustá esto a la ruta real donde clonaste tu proyecto (el que contiene este notebook, subido a GitHub).

In [ ]:
import os

REPO_DIR = os.path.expanduser('~/proyectos/Mujoco-Escenarios')  # AJUSTAR si corresponde
LOCO_MUJOCO_REPO_DIR = os.path.expanduser('~/loco-mujoco')      # carpeta donde clonaste loco-mujoco (scripts de entrenamiento)

CHECKPOINT_DIR = os.path.join(REPO_DIR, 'checkpoints')
VIDEO_DIR = os.path.join(REPO_DIR, 'videos')
STATE_FILE = os.path.join(REPO_DIR, 'training_state.json')

for d in [CHECKPOINT_DIR, VIDEO_DIR]:
    os.makedirs(d, exist_ok=True)

print('Repo del proyecto:', REPO_DIR)
print('Repo de LocoMuJoCo (scripts):', LOCO_MUJOCO_REPO_DIR)


## 1. Variables de entorno de GPU (fijar ANTES de cualquier import de jax)

`XLA_PYTHON_CLIENT_PREALLOCATE=false` es el fix que confirmamos que hacía falta en el diagnóstico -- sin esto, JAX intenta reservar de entrada un bloque grande de VRAM y falla incluso para cálculos chicos en una placa de 4GB.

In [ ]:
import os

os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'
os.environ['WANDB_MODE'] = 'disabled'
os.environ['HYDRA_FULL_ERROR'] = '1'

print('Variables de entorno seteadas')


## 2. Verificación rápida (repite el diagnóstico ya confirmado, para esta sesión de Jupyter)

In [ ]:
import jax, flax, numpy
print('jax:', jax.__version__)
print('flax:', flax.__version__)
print('numpy:', numpy.__version__)
print('Dispositivos JAX:', jax.devices())
assert any(d.platform == 'gpu' for d in jax.devices()), 'JAX no ve la GPU'

import jax.numpy as jnp
x = jnp.ones((1000, 1000))
y = jnp.dot(x, x)
y.block_until_ready()
print('Cómputo real en GPU OK. Device:', y.device)


In [ ]:
import loco_mujoco
from loco_mujoco.environments import LocoEnv
from loco_mujoco.task_factories import ImitationFactory, LAFAN1DatasetConf

envs = [n for n in LocoEnv.registered_envs.keys() if 'G1' in n or 'Unitree' in n]
print('Entornos G1/Unitree disponibles:', envs)


## 3. Entrenamiento por etapas idempotentes

Misma lógica que en la versión CPU: cada etapa entrena desde cero hasta una meta de steps, se guarda con nombre único, y si el proceso se corta, volver a correr esta celda saltea las etapas ya completas.

**Cambios respecto a la versión CPU:**
- `NUM_ENVS` bajo por defecto (memoria limitada) -- referencia: en la T4 de Colab (15GB) usábamos 256; acá arrancamos mucho más bajo y medimos.
- `use_mjwarp=false` (el backend experimental que nos rompía con `TypeError: make_data()` en Colab -- mismo problema puede aparecer acá, lo dejamos desactivado preventivamente).
- Uso de `subprocess.Popen` en vez de `os.system`, para que interrumpir el kernel mate el proceso correctamente (aprendido de la sesión anterior).

In [ ]:
STAGES = [1_000_000, 5_000_000, 20_000_000, 50_000_000]  # ajustar según lo que midas en la etapa mas chica

NUM_ENVS = 32  # punto de partida conservador para 4GB VRAM -- subir de a poco si sobra memoria

print('num_envs a usar:', NUM_ENVS)
print('Etapas planeadas:', STAGES)


In [ ]:
import json, glob, shutil, time, subprocess

def stage_checkpoint_path(steps):
    return os.path.join(CHECKPOINT_DIR, f'g1_walk_{steps}steps.pkl')

def load_state():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE) as f:
            return json.load(f)
    return {'completed_stages': []}

def save_state(state):
    with open(STATE_FILE, 'w') as f:
        json.dump(state, f, indent=2)

state = load_state()
print('Etapas ya completas según el estado guardado:', state['completed_stages'])


In [ ]:
training_dir = os.path.join(LOCO_MUJOCO_REPO_DIR, 'examples', 'training_examples', 'jax_rl_mimic')
os.chdir(training_dir)
print('Corriendo desde:', os.getcwd())

for steps in STAGES:
    ckpt_path = stage_checkpoint_path(steps)

    if steps in state['completed_stages'] and os.path.exists(ckpt_path):
        print(f'Etapa {steps} steps ya completa (checkpoint encontrado), saltando...')
        continue

    print(f'=== Entrenando etapa: {steps} steps ===')
    t0 = time.time()

    cmd = [
        'python', 'experiment.py',
        'experiment.env_params.env_name=MjxUnitreeG1',
        'experiment.env_params.use_mjwarp=false',
        'experiment.task_factory.params.lafan1_dataset_conf.dataset_name=[walk1_subject1]',
        f'experiment.total_timesteps={steps}',
        f'experiment.num_envs={NUM_ENVS}',
    ]

    proc = subprocess.Popen(cmd)
    try:
        exit_code = proc.wait()
    except KeyboardInterrupt:
        print('Interrupción recibida, matando el proceso de entrenamiento...')
        proc.terminate()
        proc.wait()
        raise

    elapsed = time.time() - t0
    print(f'Etapa {steps} steps terminada en {elapsed/60:.1f} min. Código de salida: {exit_code}')

    if exit_code != 0:
        print(f'ADVERTENCIA: la etapa {steps} terminó con error o fue interrumpida.')
        print('Si fue por memoria (CUDA_ERROR_OUT_OF_MEMORY), bajar NUM_ENVS y volver a correr esta celda.')
        break

    pkl_files = glob.glob('./outputs/**/*.pkl', recursive=True)
    if not pkl_files:
        print(f'ADVERTENCIA: no se encontró .pkl para la etapa {steps}.')
        break

    latest = max(pkl_files, key=os.path.getmtime)
    shutil.copy(latest, ckpt_path)
    print('Checkpoint guardado en:', ckpt_path)

    state['completed_stages'].append(steps)
    save_state(state)

os.chdir(REPO_DIR)
print('Loop de entrenamiento por etapas terminado. Etapas completas:', state['completed_stages'])


**Si se corta:** volvé a correr la celda de arriba tal cual -- lee `training_state.json` y saltea automáticamente las etapas ya guardadas.

**Si falla por `CUDA_ERROR_OUT_OF_MEMORY`:** bajá `NUM_ENVS` (probá 16, después 8 si hace falta) en la celda de configuración y volvé a correr -- con 4GB de VRAM es el ajuste más probable que necesites.

## 4. Evaluación del mejor checkpoint disponible

In [ ]:
state = load_state()
if not state['completed_stages']:
    print('Todavía no hay ninguna etapa completa.')
else:
    best_steps = max(state['completed_stages'])
    best_ckpt = stage_checkpoint_path(best_steps)
    print(f'Mejor checkpoint disponible: {best_ckpt} ({best_steps} steps)')


In [ ]:
os.chdir(training_dir)
!python eval.py --help


In [ ]:
# Ajustar los flags según el --help de arriba si no coinciden
!python eval.py --path "$best_ckpt"
os.chdir(REPO_DIR)


## 5. Subir el resultado a tu repositorio de GitHub

```bash
cd ~/proyectos/Mujoco-Escenarios   # ajustar a REPO_DIR
git lfs install
git lfs track "checkpoints/**"
git add .gitattributes checkpoints/ training_state.json
git commit -m "Agregar policy G1 entrenada con GPU local (WSL, GTX 1650)"
git push
```